In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path("../data/raw")
PROCESSED_PATH = Path("../data/processed")

PROCESSED_PATH.mkdir(exist_ok=True)

# Load required datasets
nav_history = pd.read_csv(RAW_PATH / "02_nav_history.csv")
investor_transactions = pd.read_csv(RAW_PATH / "08_investor_transactions.csv")
scheme_performance = pd.read_csv(RAW_PATH / "07_scheme_performance.csv")

In [11]:
print(nav_history.shape)

display(nav_history.head())

nav_history.info()

nav_history.isnull().sum()

print("Duplicate Rows:", nav_history.duplicated().sum())

(46000, 3)


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  object 
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ MB
Duplicate Rows: 0


In [12]:
nav_history['date'] = pd.to_datetime(
    nav_history['date'],
    
)

In [13]:
nav_history = nav_history.sort_values(
    ['amfi_code', 'date']
)

In [14]:
before = nav_history.shape[0]

nav_history = nav_history.drop_duplicates()

after = nav_history.shape[0]

print("Duplicates Removed:", before - after)

Duplicates Removed: 0


In [15]:
nav_history['nav'] = (
    nav_history
    .groupby('amfi_code')['nav']
    .ffill()
)

In [16]:
invalid_nav = nav_history[
    nav_history['nav'] <= 0
]

print("Invalid NAV Records:", len(invalid_nav))

Invalid NAV Records: 0


In [17]:
nav_history.to_csv(
    PROCESSED_PATH / "nav_history_clean.csv",
    index=False
)

In [18]:
print(investor_transactions.shape)

display(investor_transactions.head())

investor_transactions.info()

investor_transactions.isnull().sum()

print(
    "Duplicate Rows:",
    investor_transactions.duplicated().sum()
)

(32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  object 
 1   transaction_date    32778 non-null  object 
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  object 
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  object 
 6   city                32778 non-null  object 
 7   city_tier           32778 non-null  object 
 8   age_group           32778 non-null  object 
 9   gender              32778 non-null  object 
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  object 
 12  kyc_status          32778 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.3+ MB
Duplicate Rows: 0


In [19]:
investor_transactions.columns.tolist()

['investor_id',
 'transaction_date',
 'amfi_code',
 'transaction_type',
 'amount_inr',
 'state',
 'city',
 'city_tier',
 'age_group',
 'gender',
 'annual_income_lakh',
 'payment_mode',
 'kyc_status']

In [20]:
investor_transactions['transaction_date'] = pd.to_datetime(
    investor_transactions['transaction_date']
)

In [21]:
print(
    investor_transactions['transaction_type']
    .value_counts()
)

transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64


In [22]:
investor_transactions['transaction_type'] = (
    investor_transactions['transaction_type']
    .astype(str)
    .str.strip()
    .str.title()
)

In [24]:
invalid_amount = investor_transactions[
    investor_transactions['amount_inr'] <= 0
]

print(
    "Invalid Amount Records:",
    len(invalid_amount)
)

Invalid Amount Records: 0


In [25]:
print(
    investor_transactions['kyc_status']
    .value_counts()
)

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64


In [26]:
investor_transactions = (
    investor_transactions
    .drop_duplicates()
)

In [27]:
investor_transactions.to_csv(
    PROCESSED_PATH /
    "investor_transactions_clean.csv",
    index=False
)

In [28]:
print(scheme_performance.shape)

display(scheme_performance.head())

scheme_performance.info()

scheme_performance.isnull().sum()

(40, 19)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ra

amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64

In [29]:
return_cols = [
    col
    for col in scheme_performance.columns
    if 'return' in col.lower()
]

return_cols

['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']

In [31]:
for col in return_cols:

    scheme_performance[col] = pd.to_numeric(
        scheme_performance[col],
        errors='coerce'
    )

In [32]:
scheme_performance[return_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   return_1yr_pct  40 non-null     float64
 1   return_3yr_pct  40 non-null     float64
 2   return_5yr_pct  40 non-null     float64
dtypes: float64(3)
memory usage: 1.1 KB


In [33]:
for col in return_cols:

    anomalies = scheme_performance[
        (
            scheme_performance[col] > 100
        )
        |
        (
            scheme_performance[col] < -100
        )
    ]

    print(col)
    print("Anomalies:", len(anomalies))

return_1yr_pct
Anomalies: 0
return_3yr_pct
Anomalies: 0
return_5yr_pct
Anomalies: 0


In [35]:
invalid_expense = scheme_performance[
    (
        scheme_performance['expense_ratio_pct'] < 0.1
    )
    |
    (
        scheme_performance['expense_ratio_pct'] > 2.5
    )
]

print(
    "Expense Ratio Issues:",
    len(invalid_expense)
)

Expense Ratio Issues: 0


In [36]:
scheme_performance = (
    scheme_performance
    .drop_duplicates()
)

In [37]:
scheme_performance.to_csv(
    PROCESSED_PATH /
    "scheme_performance_clean.csv",
    index=False
)

In [38]:
print("="*50)
print("DATA CLEANING SUMMARY")
print("="*50)

print(
    "NAV History Shape:",
    nav_history.shape
)

print(
    "Investor Transactions Shape:",
    investor_transactions.shape
)

print(
    "Scheme Performance Shape:",
    scheme_performance.shape
)

DATA CLEANING SUMMARY
NAV History Shape: (46000, 3)
Investor Transactions Shape: (32778, 13)
Scheme Performance Shape: (40, 19)


## Day 2 Data Cleaning Summary

### nav_history.csv
- Converted dates to datetime
- Sorted by AMFI code and date
- Removed duplicate rows
- Forward-filled missing NAV values
- Validated NAV > 0

### investor_transactions.csv
- Standardized transaction types
- Converted dates to datetime
- Validated transaction amounts
- Checked KYC status values
- Removed duplicates

### scheme_performance.csv
- Converted return columns to numeric
- Flagged extreme return anomalies
- Validated expense ratio range (0.1%–2.5%)
- Removed duplicates

### Output
Cleaned datasets saved in:
data/processed/